**Trabalho 1 IA - JARVIS**

Implementação das funcionalidades:

- 3.1 Consulta a materiais de estudo (RAG)
- 3.2 Agenda acadêmica
- 3.3 Lista de tarefas


In [1]:
# INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS

!pip install sentence-transformers faiss-cpu openai

import os
import re
import json
import unicodedata
import numpy as np
import faiss
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer
from openai import OpenAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 33.3 MB/s eta 0:00:00


In [2]:
# CRIAÇÃO DA PASTA DATA - ARMAZENAMENTO DOS MATERIAIS

os.makedirs("data", exist_ok=True)

print("Pasta data criada")

Pasta data criada


**Observação sobre os arquivos TXT**

Após executar a célula de criação das pastas, inserir os arquivos `.txt` dos materiais acadêmicos dentro da pasta `data`. Somente depois disso executar as demais células.

In [3]:
# CRIAÇÃO DOS ARQUIVOS BASE

agenda_inicial = [
    {
        "data": "2026-05-18",
        "hora": "18:30",
        "tipo": "aula",
        "disciplina": "Inteligência Artificial",
        "descricao": "Aula sobre RAG e embeddings"
    },
    {
        "data": "2026-05-26",
        "hora": "20:30",
        "tipo": "prova",
        "disciplina": "Computação Distribuída",
        "descricao": "Prova sobre protocolos de comunicação e escalabilidade"
    }
]

tarefas_iniciais = [
    {
        "id": 1,
        "descricao": "Estudar embeddings",
        "disciplina": "Inteligência Artificial",
        "prazo": "2026-05-16",
        "concluida": False
    },
    {
        "id": 2,
        "descricao": "Revisar slides de escalabilidade",
        "disciplina": "Computação Distribuída",
        "prazo": "2026-05-23",
        "concluida": False
    }
]

with open("agenda.json", "w", encoding="utf-8") as f:
    json.dump(agenda_inicial, f, ensure_ascii=False, indent=4)

with open("tarefas.json", "w", encoding="utf-8") as f:
    json.dump(tarefas_iniciais, f, ensure_ascii=False, indent=4)

In [4]:
# FUNÇÃO: CONSULTAR AGENDA

def consultar_agenda(data_consulta=None):

    try:
        with open("agenda.json", "r", encoding="utf-8") as f:
            agenda = json.load(f)

        if data_consulta is None:
            data_consulta = datetime.now().strftime("%Y-%m-%d")

        data_formatada = datetime.strptime(
            data_consulta,
            "%Y-%m-%d"
        ).strftime("%d/%m/%Y")

        eventos = [
            evento for evento in agenda
            if evento["data"] == data_consulta
        ]

        if not eventos:
            return f"Nenhum evento encontrado para o dia {data_formatada}"

        resposta = ""

        for evento in eventos:
            resposta += (
                f"Agenda {data_formatada}\n"
                f"{evento['tipo'].capitalize()}: {evento['descricao']}\n"
                f"Disciplina: {evento['disciplina']}\n"
                f"Horário: {evento['hora']}\n\n"
            )

        return resposta

    except Exception as e:
        return f"Erro ao consultar agenda: {str(e)}"

In [5]:
# FUNÇÃO: LISTAR TAREFAS

def listar_tarefas():

    try:
        with open("tarefas.json", "r", encoding="utf-8") as f:
            tarefas = json.load(f)

        if not tarefas:
            return "Nenhuma tarefa cadastrada"

        resposta = ""

        for tarefa in tarefas:

            status = "Concluída" if tarefa["concluida"] else "Pendente"

            prazo_formatado = datetime.strptime(
                tarefa["prazo"],
                "%Y-%m-%d"
            ).strftime("%d/%m/%Y")

            resposta += (
                f"Tarefa {tarefa['id']}\n"
                f"Descrição: {tarefa['descricao']}\n"
                f"Disciplina: {tarefa['disciplina']}\n"
                f"Prazo: {prazo_formatado}\n"
                f"Status: {status}\n\n"
            )

        return resposta

    except Exception as e:
        return f"Erro ao listar tarefas: {str(e)}"

In [6]:
# FUNÇÃO: ADICIONAR TAREFA

def adicionar_tarefa(descricao, disciplina, prazo):

    try:
        with open("tarefas.json", "r", encoding="utf-8") as f:
            tarefas = json.load(f)

        novo_id = max([t["id"] for t in tarefas], default=0) + 1

        nova_tarefa = {
            "id": novo_id,
            "descricao": descricao,
            "disciplina": disciplina,
            "prazo": prazo,
            "concluida": False
        }

        tarefas.append(nova_tarefa)

        with open("tarefas.json", "w", encoding="utf-8") as f:
            json.dump(tarefas, f, ensure_ascii=False, indent=4)

        prazo_formatado = datetime.strptime(
            prazo,
            "%Y-%m-%d"
        ).strftime("%d/%m/%Y")

        return (
            "Tarefa adicionada\n\n"
            f"ID: {novo_id}\n"
            f"Descrição: {descricao}\n"
            f"Disciplina: {disciplina}\n"
            f"Prazo: {prazo_formatado}"
        )

    except Exception as e:
        return f"Erro ao adicionar tarefa: {str(e)}"

In [7]:
# FUNÇÃO: CONCLUIR TAREFA

def concluir_tarefa(id_tarefa):

    try:
        with open("tarefas.json", "r", encoding="utf-8") as f:
            tarefas = json.load(f)

        tarefa_encontrada = None

        for tarefa in tarefas:
            if tarefa["id"] == id_tarefa:
                tarefa["concluida"] = True
                tarefa_encontrada = tarefa
                break

        if tarefa_encontrada is None:
            return f"Nenhuma tarefa encontrada com o ID {id_tarefa}."

        with open("tarefas.json", "w", encoding="utf-8") as f:
            json.dump(tarefas, f, ensure_ascii=False, indent=4)

        return (
            "Tarefa concluída\n\n"
            f"ID: {tarefa_encontrada['id']}\n"
            f"Descrição: {tarefa_encontrada['descricao']}"
        )

    except Exception as e:
        return f"Erro ao concluir tarefa: {str(e)}"

In [8]:
# LEITURA E TRATAMENTO DOS ARQUIVOS

documentos = []

def limpar_texto(texto):

    texto = texto.replace("\ufeff", "")
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.replace("\u00a0", " ")
    texto = texto.replace("\t", " ")
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"\s+([.,;:!?])", r"\1", texto)
    texto = texto.strip()

    return texto

arquivos_txt = [
    arquivo for arquivo in os.listdir("data")
    if arquivo.endswith(".txt")
]

if not arquivos_txt:
    print("Nenhum arquivo encontrado na pasta data")

else:
    for arquivo in arquivos_txt:

        caminho_txt = os.path.join("data", arquivo)

        with open(caminho_txt, "r", encoding="utf-8") as f:
            texto_completo = f.read()

        texto_completo = limpar_texto(texto_completo)

        documentos.append({
            "arquivo": arquivo,
            "texto": texto_completo
        })

    print(f"{len(documentos)} documentos carregados")

3 documentos carregados


In [9]:
# DIVISÃO DOS TEXTOS EM CHUNKS

def dividir_em_chunks(texto, tamanho_maximo=1000):

    frases = texto.split(". ")
    chunks = []
    chunk_atual = ""

    for frase in frases:

        frase = frase.strip()

        if not frase:
            continue

        if not frase.endswith("."):
            frase += "."

        if len(chunk_atual) + len(frase) <= tamanho_maximo:
            chunk_atual += " " + frase
        else:
            if chunk_atual.strip():
                chunks.append(chunk_atual.strip())
            chunk_atual = frase

    if chunk_atual.strip():
        chunks.append(chunk_atual.strip())

    return chunks

chunks = []

for documento in documentos:

    partes = dividir_em_chunks(
        texto=documento["texto"]
    )

    for i, parte in enumerate(partes):
        chunks.append({
            "arquivo": documento["arquivo"],
            "chunk_id": i,
            "texto": parte
        })

print(f"{len(chunks)} chunks criados")

22 chunks criados


In [10]:
# CRIAÇÃO DOS EMBEDDINGS E ÍNDICE FAISS

modelo_embeddings = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

textos_chunks = [chunk["texto"] for chunk in chunks]

embeddings = modelo_embeddings.encode(
    textos_chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

dimensao = embeddings.shape[1]

indice_faiss = faiss.IndexFlatL2(dimensao)
indice_faiss.add(embeddings)

print("Embeddings e índice FAISS criados")
print("Quantidade de chunks:", indice_faiss.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings e índice FAISS criados
Quantidade de chunks: 22


In [13]:
# CONEXÃO COM A API DA GEMMA

# Por motivos de segurança, o token da API não foi incluído no repositório
# Antes de executar o sistema, substitua abaixo pelo token original

TOKEN_GEMMA = "COLOCAR_TOKEN"

client = OpenAI(
    base_url="https://llm.liaufms.org/v1/gemma-3-12b-it",
    api_key=TOKEN_GEMMA,
    timeout=60
)

In [14]:
# FUNÇÃO: RESPONDER COM RAG

def responder_com_rag(pergunta, top_k=3):

    try:
        embedding_pergunta = modelo_embeddings.encode(
            [pergunta],
            convert_to_numpy=True
        )

        distancias, indices = indice_faiss.search(
            embedding_pergunta,
            top_k
        )

        contexto = ""

        for indice in indices[0]:

            chunk = chunks[indice]

            contexto += (
                f"\nFonte: {chunk['arquivo']}\n"
                f"{chunk['texto']}\n"
            )

        prompt = f"""
Você é um assistente acadêmico.

Responda à pergunta do usuário utilizando somente as informações presentes no contexto abaixo.
Se a informação não estiver no contexto, diga que não encontrou informações suficientes.

Contexto:
{contexto}

Pergunta:
{pergunta}
"""

        resposta = client.chat.completions.create(
            model="google/gemma-3-12b-it",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return resposta.choices[0].message.content

    except Exception as e:
        return f"Erro ao responder com RAG: {str(e)}"

In [16]:
# TESTES FINAIS

print("TESTE - RAG")
print(responder_com_rag("Explique o que é KNN"))

print("\n" + "=" * 80)

print("TESTE - AGENDA")
print(consultar_agenda("2026-05-26"))

print("\n" + "=" * 80)

print("TESTE - LISTA DE TAREFAS")
print(listar_tarefas())

print("\n" + "=" * 80)

print("TESTE - ADICIONAR TAREFA")
print(adicionar_tarefa("Revisar árvores de decisão", "Inteligência Artificial", "2026-05-28"))

print("\n" + "=" * 80)

print("TESTE - CONCLUIR TAREFA")
print(concluir_tarefa(1))

TESTE - RAG
O algoritmo K-Nearest Neighbors (KNN), também conhecido como k-Vizinhos Mais Próximos, é um método de aprendizado de máquina supervisionado utilizado em tarefas de classificação e regressão. Seu funcionamento é baseado no conceito de similaridade entre exemplos, considerando que objetos semelhantes tendem a apresentar comportamentos semelhantes ou pertencer à mesma classe. O KNN armazena os exemplos conhecidos e realiza os cálculos somente quando um novo exemplo precisa ser analisado ou classificado. No problema de classificação, o novo exemplo recebe a classe mais frequente entre os vizinhos selecionados.

TESTE - AGENDA
Agenda 26/05/2026
Prova: Prova sobre protocolos de comunicação e escalabilidade
Disciplina: Computação Distribuída
Horário: 20:30



TESTE - LISTA DE TAREFAS
Tarefa 1
Descrição: Estudar embeddings
Disciplina: Inteligência Artificial
Prazo: 16/05/2026
Status: Concluída

Tarefa 2
Descrição: Revisar slides de escalabilidade
Disciplina: Computação Distribuída
